This file is used to test whether free generation or chat templates are more suitable for math reasoning tasks. Current results show chat templates are better for stronger models.

Tested models: Llama 3.1 8B instrcut; 3.2 3B instruct and 3.2 1B instrctu


Should use free generation instead of chat templates for llama-3.2-1B-Instruct ?

Should use chat template instead of free generation for llama-3.2-3B-Instruct

Should use chat template instead of free generation for llama-3.1-8B-Instruct


In [1]:
from vllm import LLM, SamplingParams
llm = LLM(model="meta-llama/Llama-3.2-1b-Instruct")


INFO 04-12 15:27:14 config.py:510] This model supports multiple tasks: {'generate', 'embed', 'classify', 'reward', 'score'}. Defaulting to 'generate'.
WARNING 04-12 15:27:14 arg_utils.py:1103] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 04-12 15:27:14 config.py:1458] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 04-12 15:27:14 llm_engine.py:234] Initializing an LLM engine (v0.6.6.post1) with config: model='meta-llama/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 04-12 15:27:20 model_runner.py:1099] Loading model weights took 14.9888 GB
INFO 04-12 15:27:20 worker.py:241] Memory profiling takes 0.72 seconds
INFO 04-12 15:27:20 worker.py:241] the current vLLM instance can use total_gpu_memory (47.43GiB) x gpu_memory_utilization (0.90) = 42.69GiB
INFO 04-12 15:27:20 worker.py:241] model weights take 14.99GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.19GiB; the rest of the memory reserved for KV Cache is 26.42GiB.
INFO 04-12 15:27:20 gpu_executor.py:76] # GPU blocks: 13529, # CPU blocks: 2048
INFO 04-12 15:27:20 gpu_executor.py:80] Maximum concurrency for 131072 tokens per request: 1.65x
INFO 04-12 15:27:23 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_util

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:16<00:00,  2.15it/s]

INFO 04-12 15:27:40 model_runner.py:1535] Graph capturing finished in 16 secs, took 0.26 GiB
INFO 04-12 15:27:40 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 20.01 seconds


In [2]:
sampling_params = llm.get_default_sampling_params()
print(sampling_params)
sampling_params.max_tokens = 2048
sampling_params.n=2
conversation = [
    {
        "role": "system",
        "content": "Please solve the following question step by step and return the final answer in \\boxed{}. Each step should start with Step:"
    },
    {
        "role": "user",
        "content": "what is the number of prime numbers between 1 and 200?"
    }
]
# using the default chat template
outputs = llm.chat(conversation, sampling_params, use_tqdm=True)
for i in range(len(outputs[0].outputs)):
    print(outputs[0].outputs[i].text)
    print("-"*100)


SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=1.0, top_p=1.0, top_k=-1, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=16, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None)
INFO 04-12 15:28:02 chat_utils.py:333] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:  50%|█████     | 1/2 [00:29<00:29, 29.27s/it, est. speed input: 2.53 toks/s, output: 58.50 toks/s]

Step 1:  To find the number of prime numbers between 1 and 200, we will use the Sieve of Eratosthenes algorithm. The Sieve of Eratosthenes is an ancient algorithm used to find all primes smaller than a given number.

Step 2:  Start by creating a list of all numbers from 1 to 200. The Sieve of Eratosthenes works by iteratively marking as composite (not prime) the multiples of each prime, starting with the first prime number, 2.

Step 3:  Begin by marking all the multiples of 2, which is the first prime number, as composite. This includes 4, 6, 8, 10, and so on.

Step 4:  Move on to the next unmarked number, which is 3. Mark all the multiples of 3, which are 6, 9, 12, 15, etc.

Step 5:  Continue this process with the next unmarked number, which is 5. Mark all the multiples of 5, which are 10, 15, 20, 25, etc.

Step 6:  Repeat this process with the next unmarked number, which is 7. Mark all the multiples of 7, which are 14, 21, 28, 35, etc.

Step 7:  Continue this process with the next un

In [8]:
len(outputs[0].prompt_token_ids)

74

In [6]:
prompt = ["solve the following problem and return the final answer in \\boxed{}: what is the number of prime numbers between 1 and 200?"]
sampling_params.max_tokens = 2048
sampling_params.n=10
outputs = llm.generate(prompt, sampling_params, use_tqdm=True)
for i in range(len(outputs[0].outputs)):
    print(outputs[0].outputs[i].text)
    print("-"*100)

Processed prompts:  10%|█         | 1/10 [00:24<03:40, 24.46s/it, est. speed input: 1.19 toks/s, output: 284.51 toks/s]

 
Step 1: first, find the number of prime numbers between 1 and 200.
Step 2: All prime numbers between 1 and 200 are: 2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199.
Step 3: Now let’s count the number of primes in the list
Step 4: There are 46 primes on this list.
Step 5: Therefore, there are 46 prime numbers between 1 and 200.

The final answer is: $\boxed{46}$
----------------------------------------------------------------------------------------------------
 
= 47\\_
In order to solve this problem, we need to determine the count of prime numbers in a given range.
The final answer is 47. \_\\_

## Step 1: Understand what a prime number is.
A prime number is a number greater than 1 that has no positive divisors other than 1 and itself.

## Step 2: Identify the range of the problem.
The problem asks for the count of prime numbers betw

In [1]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

max_model_len, tp_size = 8192, 1
model_name = "meta-llama/Llama-3.2-3B-Instruct"
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = LLM(model=model_name, tensor_parallel_size=tp_size, max_model_len=max_model_len, trust_remote_code=True, enforce_eager=True)
sampling_params = llm.get_default_sampling_params()
messages_list = [
    [{"role": "system", 
      "content": "You are a helpful assistant.Solve the following problem step by step and return the final answer in \\boxed{}:",
      },
      {"role": "user", 
      "content": "What is the number of prime numbers between 1 and 200?"},
      {"role": "assistant", 
      "content": "Step 1: Find all prime numbers between 1 and 200."},]

]

prompt_token_ids = [tokenizer.apply_chat_template(messages, add_generation_prompt=True) for messages in messages_list]
prompt_token_ids[0] = prompt_token_ids[0][:-5] # remove the last 5 tokens for chat completion
outputs = llm.generate(prompt_token_ids=prompt_token_ids, sampling_params=sampling_params)

generated_text = [output.outputs[0].text for output in outputs]
print(generated_text)

INFO 04-13 15:21:40 config.py:510] This model supports multiple tasks: {'score', 'generate', 'reward', 'classify', 'embed'}. Defaulting to 'generate'.
WARNING 04-13 15:21:40 cuda.py:98] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 04-13 15:21:40 config.py:642] Async output processing is not supported on the current platform type cuda.
INFO 04-13 15:21:40 llm_engine.py:234] Initializing an LLM engine (v0.6.6.post1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto, qua

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 04-13 15:21:45 model_runner.py:1099] Loading model weights took 14.2487 GB
INFO 04-13 15:21:47 worker.py:241] Memory profiling takes 1.40 seconds
INFO 04-13 15:21:47 worker.py:241] the current vLLM instance can use total_gpu_memory (47.43GiB) x gpu_memory_utilization (0.90) = 42.69GiB
INFO 04-13 15:21:47 worker.py:241] model weights take 14.25GiB; non_torch_memory takes 0.10GiB; PyTorch activation peak memory takes 1.44GiB; the rest of the memory reserved for KV Cache is 26.89GiB.
INFO 04-13 15:21:47 gpu_executor.py:76] # GPU blocks: 31472, # CPU blocks: 4681
INFO 04-13 15:21:47 gpu_executor.py:80] Maximum concurrency for 8192 tokens per request: 61.47x
INFO 04-13 15:21:52 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 6.42 seconds


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.58it/s, est. speed input: 178.60 toks/s, output: 41.41 toks/s]

[' We need to list all numbers that are only divisible by 1 and themselves.']


In [3]:
tokenizer.batch_decode(prompt_token_ids)

['<|im_start|>system\nYou are a helpful assistant.Solve the following problem step by step and return the final answer in \\boxed{}:<|im_end|>\n<|im_start|>user\nWhat is the number of prime numbers between 1 and 200?<|im_end|>\n<|im_start|>assistant\nStep 1: Find all prime numbers between 1 and 200.']

: 